## Imports

In [27]:
import pandas as pd
import numpy as np

In [6]:
""" Load SP500 data """

sp_500_df = pd.read_csv(
    "../../data/raw/sp500_2010_historical.csv",
    index_col=0,      # first column is the date
    parse_dates=True  # parse dates automatically
)

# Check it loaded correctly
print(sp_500_df.head())
print(sp_500_df.tail())
print(sp_500_df.shape)

                  Close         High          Low         Open      Volume
Date                                                                      
2010-01-04  1132.989990  1133.869995  1116.560059  1116.560059  3991400000
2010-01-05  1136.520020  1136.630005  1129.660034  1132.660034  2491020000
2010-01-06  1137.140015  1139.189941  1133.949951  1135.709961  4972660000
2010-01-07  1141.689941  1142.459961  1131.319946  1136.270020  5270680000
2010-01-08  1144.979980  1145.390015  1136.219971  1140.520020  4389590000
                  Close         High          Low         Open      Volume
Date                                                                      
2026-06-12  7431.459961  7456.399902  7363.009766  7410.850098  4950530000
2026-06-15  7554.290039  7577.919922  7516.750000  7516.750000  5674670000
2026-06-16  7511.350098  7564.959961  7508.680176  7548.779785  5286210000
2026-06-17  7420.100098  7532.169922  7402.609863  7524.500000  5883740000
2026-06-18  7500.580078  

In [4]:
""" Load market data """

market_df = pd.read_csv(
    "../../data/raw/fixed_market_data.csv",
    index_col=0, 
    sep=';'     
)

# Check it loaded correctly
print(market_df.head())
print(market_df.tail())
print(market_df.shape)

          EMP    PE    CAPE    DY     Rho   MOV     IR      RR    Y02    Y10  \
Date                                                                           
4/10/88  2,0%  12,9  14,469   3,6  -0,083    118  6,02  -1,061  7,459  8,484   
4/17/88  2,0%  12,4   13,96  3,75  -0,078    121  5,88   -0,76  7,582  8,737   
4/24/88  2,0%  12,4   13,95  3,75  -0,051    123  5,83   -0,76  7,618  8,773   
5/1/88   2,0%  12,5  14,036  3,75  -0,054  124,2  5,98   -0,76  7,728  8,873   
5/8/88   2,0%  12,3  13,761  3,82  -0,079  118,4  6,29   -0,76  7,885   8,99   

         ...       YSS          NYF     _AU   _DXY    _LCP     _TY   _OIL  \
Date     ...                                                                
4/10/88  ...  0,251622  6,896895065   450,5  89,14  2473,7   97,99  16,87   
4/17/88  ...  0,328976  2,631748517  456,25  88,31  2328,2  96,534  18,32   
4/24/88  ...  0,156475  2,631748517  449,25  88,89  2201,4   96,47  18,13   
5/1/88   ...  0,146747  2,631748517     449  89,16  21

## Feature Engineering

In [ ]:
# Log returns to normalize data
level_cols = ['Close', 'High', 'Low', 'Open']
for col in level_cols:
        if col in sp_500_df.columns:
            # ln(P_t / P_t-1)
            sp_500_df[f'{col}_LogRet'] = np.log(sp_500_df[col] / sp_500_df[col].shift(1))

sp_500_df.dropna(inplace=True)

sp_500_df.head()

,Close,High,Low,Open,Volume,target,target_up,Close_LogRet,High_LogRet,Low_LogRet,Open_LogRet
Date,,,,,,,,,,,
2010-01-05,1136.520020,1136.630005,1129.660034,1132.660034,2491020000,-0.064592,0,0.003111,0.002431,0.011664,0.014316
2010-01-06,1137.140015,1139.189941,1133.949951,1135.709961,4972660000,-0.062393,0,0.000545,0.002250,0.003790,0.002689
2010-01-07,1141.689941,1142.459961,1131.319946,1136.270020,5270680000,-0.074407,0,0.003993,0.002866,-0.002322,0.000493
2010-01-08,1144.979980,1145.390015,1136.219971,1140.520020,4389590000,-0.065032,0,0.002878,0.002561,0.004322,0.003733
2010-01-11,1146.979980,1149.739990,1142.020020,1145.959961,4255780000,-0.068746,0,0.001745,0.003791,0.005092,0.004758


In [ ]:
# ---------------------------------------------------------
# 2. Base Stationarity (Log Returns for Assets)
# ---------------------------------------------------------
asset_cols = ['_AU', '_DXY', '_LCP', '_TY', '_OIL', '_MKT', '_VA', '_GR']
for col in asset_cols:
    if col in market_df.columns:
        market_df[f'{col}_LogRet'] = np.log(market_df[col] / market_df[col].shift(1))

# ---------------------------------------------------------
# 3. CAPE & USD (Currency Squeeze Features)
# ---------------------------------------------------------
if 'CAPE' in market_df.columns and '_DXY' in market_df.columns:
    # Feature A: CAPE Earnings Yield (Inverse of CAPE)
    market_df['CAPE_Yield_Pct'] = (1 / market_df['CAPE']) * 100
    
    # Feature B: USD 3-Month Momentum (Is the dollar surging?)
    # ~63 trading days in 3 months
    market_df['USD_Momentum_3M'] = market_df['_DXY'].pct_change(63)
    
    # Feature C: Valuation/Currency Squeeze Interaction
    # Multiplies CAPE by USD Momentum. 
    # High CAPE * High USD positive momentum = High Risk Score
    market_df['Valuation_Currency_Risk'] = market_df['CAPE'] * market_df['USD_Momentum_3M']

# ---------------------------------------------------------
# 4. Cross-Asset Ratios & Spreads
# ---------------------------------------------------------
# Copper/Gold Ratio (Growth proxy)
if '_LCP' in market_df.columns and '_AU' in market_df.columns:
    market_df['Copper_Gold_Ratio'] = market_df['_LCP'] / market_df['_AU']
    market_df['Copper_Gold_Trend_3M'] = market_df['Copper_Gold_Ratio'].pct_change(63) 
    
# Value vs Growth Regime
if '_VA' in market_df.columns and '_GR' in market_df.columns:
    market_df['Value_Growth_Ratio'] = market_df['_VA'] / market_df['_GR']
    
# Equity Risk Premium (ERP)
if 'PE' in market_df.columns and 'Y10' in market_df.columns:
    market_df['Earnings_Yield_Pct'] = (1 / market_df['PE']) * 100
    market_df['ERP'] = market_df['Earnings_Yield_Pct'] - market_df['Y10']

# ---------------------------------------------------------
# 5. Macro Acceleration (3-Month Differences)
# ---------------------------------------------------------
macro_yoy_cols = ['CPI', 'UN', 'GDP', 'M2', 'EMP', 'MG']
for col in macro_yoy_cols:
    if col in market_df.columns:
        market_df[f'{col}_Acceleration_3M'] = market_df[col] - market_df[col].shift(63)

# ---------------------------------------------------------
# 6. Financial Stress Regimes (Rolling Z-Scores)
# ---------------------------------------------------------
# 1-year (252 day) rolling Z-score for risk metrics
risk_cols = ['MOV', 'YSS', 'NYF']
for col in risk_cols:
    if col in market_df.columns:
        rolling_mean = market_df[col].rolling(window=252).mean()
        rolling_std = market_df[col].rolling(window=252).std()
        market_df[f'{col}_ZScore_1Y'] = (market_df[col] - rolling_mean) / rolling_std

# ---------------------------------------------------------
# 7. Clean Up & Finalize
# ---------------------------------------------------------
# Drop rows with NaNs caused by the 1-year rolling windows and 3-month lags
final_macro = df_feat.dropna()
final_macro.head()

TypeError: unsupported operand type(s) for /: 'int' and 'str'

## Aggregation/Windowing

In [33]:
# ---------------------------------------------------------
# 1. Price Action & Volatility Features
# ---------------------------------------------------------
# Daily trading range (proxy for intraday volatility)
sp_500_df['Intraday_Range'] = sp_500_df['High_LogRet'] - sp_500_df['Low_LogRet']
    
# Intraday direction (Close relative to Open)
sp_500_df['Intraday_Return'] = sp_500_df['Close_LogRet'] - sp_500_df['Open_LogRet']

# Rolling 5-day and 21-day realized volatility
sp_500_df['Vol_5d'] = sp_500_df['Close_LogRet'].rolling(window=5).std()
sp_500_df['Vol_21d'] = sp_500_df['Close_LogRet'].rolling(window=21).std()
    
# Cumulative returns over the last week and month
sp_500_df['Return_5d'] = sp_500_df['Close_LogRet'].rolling(window=5).sum()
sp_500_df['Return_21d'] = sp_500_df['Close_LogRet'].rolling(window=21).sum()

# ---------------------------------------------------------
# 2. Volume Features
# ---------------------------------------------------------
# Volume day-over-day percentage change
sp_500_df['Volume_Pct_Change'] = sp_500_df['Volume'].pct_change()
    
# Volume relative to its 21-day moving average (Volume Surge)
vol_ma_21 = sp_500_df['Volume'].rolling(window=21).mean()
sp_500_df['Volume_Surge_Ratio'] = sp_500_df['Volume'] / vol_ma_21

# ---------------------------------------------------------
# 3. Time Series Lags (Memory)
# ---------------------------------------------------------
# Lags of the close return (yesterday, 2 days ago, 3 days ago)
for i in [1, 2, 3]:
    sp_500_df[f'Close_LogRet_Lag{i}'] = sp_500_df['Close_LogRet'].shift(i)
    sp_500_df[f'Volume_Pct_Change_Lag{i}'] = sp_500_df['Volume_Pct_Change'].shift(i)

# ---------------------------------------------------------
# 4. Clean Up
# ---------------------------------------------------------
# Drop rows with NaNs created by rolling windows and lags
sp_500_df = sp_500_df.dropna()
sp_500_df.head()


,Close,High,Low,Open,Volume,target,target_up,Close_LogRet,High_LogRet,Low_LogRet,...,Return_5d,Return_21d,Volume_Pct_Change,Volume_Surge_Ratio,Close_LogRet_Lag1,Volume_Pct_Change_Lag1,Close_LogRet_Lag2,Volume_Pct_Change_Lag2,Close_LogRet_Lag3,Volume_Pct_Change_Lag3
Date,,,,,,,,,,,,,,,,,,,,,
2011-11-30,1246.959961,1247.109985,1196.719971,1196.719971,5801910000,0.008533,1,0.042403,0.035454,0.004120,...,-0.009498,-0.013426,0.346085,1.389020,-0.025049,-0.024131,-0.025291,-0.161556,0.004910,0.040830
2011-12-30,1257.599976,1264.119995,1257.459961,1262.819946,2271850000,0.052871,1,-0.004301,0.000459,0.006150,...,-0.007327,-0.019131,-0.608431,0.554475,0.042403,0.346085,-0.025049,-0.024131,-0.025291,-0.161556
2012-01-31,1312.410034,1321.410034,1306.689941,1313.530029,4235550000,0.046997,1,-0.000457,0.003981,0.004756,...,-0.012694,-0.016310,0.864362,1.036740,-0.004301,-0.608431,0.042403,0.346085,-0.025049,-0.024131
2012-02-29,1365.680054,1378.040039,1363.810059,1372.199951,4482370000,0.027532,1,-0.004748,0.003599,-0.001582,...,0.007849,-0.004270,0.058273,1.117551,-0.000457,0.864362,-0.004301,-0.608431,0.042403,0.346085
2012-03-30,1408.469971,1410.890015,1401.420044,1403.310059,3676890000,-0.001881,0,0.003692,0.003906,0.007061,...,0.036589,0.009586,-0.179700,0.932112,-0.004748,0.058273,-0.000457,0.864362,-0.004301,-0.608431


## Model Feature Importance